# Backtest auditável — Baseline V0 (Robo Abertura WDO v1.35)

Notebook de **pesquisa, validação e relatório**. Toda a lógica (dados, indicadores, estratégia, motor, métricas) vive no pacote `wdo` (`src/wdo/`); aqui apenas se configura o cenário, executa e se visualizam os resultados. Não é equivalente tick-a-tick ao Strategy Tester: usa candles M5 e a política OHLC conservadora.

**Política de dados:** este notebook só acessa a partição de **pesquisa** (`configs/wdo_default.toml`, `[partitions]`). Validação e holdout exigem autorização explícita e não devem ser abertos aqui.


## 1. Overview e auditoria da estratégia

**Inputs do EA preservados:** símbolo, lote, gain/loss, offset e modo de canal, rompimento, magic, desvio, tolerância EMA, RSI(7), faixas RSI e janela 09:00–10:30.

**Indicadores (point-in-time):** EMA 13/17/21 em H1 e D1 sobre o **último candle fechado**; RSI(7) M5 até a barra anterior; PDH/PDL do dia anterior. O EA lê EMAs H1/D1 ainda em formação: esta é uma aproximação conservadora e documentada, não uma réplica.

**Execução (motor v1.0.0):** a estratégia só vê a abertura da barra corrente e barras fechadas; gatilhos por toque de nível executam **no nível** (ou na abertura, se já houve gap), com slippage contra e dentro do range OHLC; a barra de entrada é processada com hipótese adversa; stops com gap executam no pior preço.

| Regra MT5 | Implementação Python |
|---|---|
| P1: toque da primeira EMA | ordem a nível na zona EMA ± tolerância, executa no nível de toque |
| P2: mínima/máxima da vela anterior | referência móvel; fade a nível, barra a barra |
| P3: stop/fade no canal | ordens pendentes stop/limit; se as duas tocarem na mesma barra, vale o pior preço |
| P4: RSI fora de PDH/PDL | IFR já conhecido: a mercado na abertura; espera: ordens a nível |
| TP/SL 6/10 pontos | níveis por trade, com política intrabar conservadora |
| uma operação/dia e fim de janela | limite diário no motor e cancelamento de pendências |

### Limitações deliberadas
- Não há ticks, bid/ask ou book: spread deve ser incorporado em `slippage_points`. Custos e slippage do cenário base são **provisórios** (não verificados).
- Se stop e alvo ocorrem na mesma barra, o padrão é o resultado adverso.
- Não há trailing stop ou break-even no EA fornecido; portanto não foram inventados.
- A série de dados aparenta ser ajustada (preços fora do tick); origem ainda não auditada.


## 2. Ambiente

Kernel **Python (wdo-backtest)** (`conda env create -f environment.yml`; o pacote `wdo` é instalado em modo editável).


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from wdo import (
    Config, Partitions, load_partition_bars, run_partition_backtest, print_backtest_report, plot_results, metrics,
)

# Caminhos relativos à raiz do repositório (o notebook vive em notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT / 'src' / 'wdo').exists(), 'Execute a partir da raiz do repositório ou de notebooks/.'


## 3. Parâmetros

Cenário e partições vêm de `configs/wdo_default.toml`. Custos, slippage e valor do ponto são premissas **provisórias** (D6): substituir pelos valores reais quando informados.


In [ ]:
config = Config.from_toml(ROOT / 'configs' / 'wdo_default.toml')
parts = Partitions.from_toml(ROOT / 'configs' / 'wdo_default.toml')
PARTITION = 'research'          # validação/holdout exigem authorized=True e autorização do usuário
config


## 4. Carga de dados (guardada por partição)

`load_partition_bars` lê o export bruto do MT5 e devolve só as barras **até o fim da partição** (o histórico anterior serve de aquecimento dos indicadores; barras posteriores nunca chegam ao motor).


In [ ]:
bars, START, END = load_partition_bars(ROOT / 'data' / 'raw' / 'wdo_data.csv', parts, PARTITION, timezone=config.timezone)
print(f'{len(bars):,} barras M5 | {bars.datetime.min()} -> {bars.datetime.max()} | janela operável {START} -> {END}')

# Outras fontes (série por contrato + rollover, Parquet, HTTP): ver wdo.data (build_continuous_contract, load_csv_or_parquet, load_http_ohlcv).


## 5. Replay, trade log e métricas

O motor atualiza indicadores com dados anteriores, processa posição e ordens em ordem cronológica e registra cada decisão em `events`.


In [ ]:
results = run_partition_backtest(bars, parts, PARTITION, config)
print(f'motor {results.engine_version} | estratégia {results.strategy_name}')
print_backtest_report(results)
trades = results.trades
events = results.events
display(trades.head(10))
display(events.head(20))


## 6. Visualizações

Equity, drawdown, PnL acumulado, distribuição, PnL mensal e trades mensais são produzidos por `plot_results`. A comparação LONG/SHORT é auditável no log e no resumo abaixo.


In [ ]:
plot_results(results)
long_short = trades.groupby('side').agg(trades=('trade_id','size'), net_pnl=('net_pnl','sum'), win_rate=('net_pnl', lambda x: x.gt(0).mean()*100))
display(long_short)


## 7. Validações e diferenças frente ao MT5

Antes de confiar nos resultados, valide: cobertura e timezone do rollover; barras ausentes; `trades` contra `equity`; política quando TP/SL coexistem na barra; e, se houver dados de tick, a divergência versus MT5. O resultado é um backtest OHLC conservador, não um relatório oficial de execução da corretora.


In [ ]:
# Reconciliação após a execução:
assert abs(trades.equity_after_trade.iloc[-1] - (config.initial_capital + trades.net_pnl.sum())) < 1e-6
assert trades.trade_id.is_unique
assert not bars.datetime.duplicated().any()
assert bars.datetime.max() <= pd.Timestamp(END, tz=config.timezone) + pd.Timedelta(days=1)   # nenhuma barra além da partição
pd.Series(metrics(results)).to_frame('valor')
